<a href="https://colab.research.google.com/github/Prachisoni07/ml/blob/main/Yolo11n__train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile

os.environ["KAGGLE_CONFIG_DIR"] = "/content"
!kaggle datasets download -d rajkumarl/people-clothing-segmentation

dataset_zip = "/content/people-clothing-segmentation.zip"
extract_path = "/content/dataset"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset extracted successfully.")
!ls /content/dataset

Dataset URL: https://www.kaggle.com/datasets/rajkumarl/people-clothing-segmentation
License(s): CC0-1.0
people-clothing-segmentation.zip: Skipping, found more recently modified local copy (use --force to force download)
✅ Dataset extracted successfully.
 jpeg_images   jpeg_masks  'labels (1).csv'   labels.csv   png_images   png_masks


In [ ]:
import pandas as pd

label_path = "/content/dataset/labels.csv"
labels_df = pd.read_csv(label_path)

print(labels_df.head())
print("Column Names:", labels_df.columns.tolist())

   Unnamed: 0   label_list
0           0          NaN
1           1  accessories
2           2          bag
3           3         belt
4           4       blazer
Column Names: ['Unnamed: 0', 'label_list']


In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from PIL import Image

image_dir = "/content/dataset/jpeg_images/IMAGES"
mask_dir = "/content/dataset/jpeg_masks/MASKS"
label_path = "/content/dataset/labels.csv"

dataset_path = "/content/dataset_yolo"
os.makedirs(dataset_path, exist_ok=True)
os.makedirs(os.path.join(dataset_path, "images", "train"), exist_ok=True)
os.makedirs(os.path.join(dataset_path, "images", "val"), exist_ok=True)
os.makedirs(os.path.join(dataset_path, "labels", "train"), exist_ok=True)
os.makedirs(os.path.join(dataset_path, "labels", "val"), exist_ok=True)

labels_df = pd.read_csv(label_path)
labels_df["label_list"] = labels_df["label_list"].fillna("background")

train_df, val_df = train_test_split(labels_df, test_size=0.2, random_state=42)

def create_mask(mask_path, class_id):
    mask = np.zeros((512, 1024), dtype=np.uint8)
    mask[:, :] = class_id
    Image.fromarray(mask).save(mask_path)

def prepare_dataset(df, image_source, image_dest, label_dest):
    for _, row in df.iterrows():
        image_name = f"img_{row['Unnamed: 0']:04d}.jpeg"
        class_id = row["Unnamed: 0"]

        src_image_path = os.path.join(image_source, image_name)
        if not os.path.exists(src_image_path):
            print(f"Warning: Image {src_image_path} not found. Skipping.")
            continue

        shutil.copy(src_image_path, os.path.join(image_dest, image_name))

        mask_name = image_name.replace(".jpeg", ".png")
        mask_path = os.path.join(label_dest, mask_name)
        create_mask(mask_path, class_id)

prepare_dataset(train_df, image_dir,
                os.path.join(dataset_path, "images", "train"),
                os.path.join(dataset_path, "labels", "train"))

prepare_dataset(val_df, image_dir,
                os.path.join(dataset_path, "images", "val"),
                os.path.join(dataset_path, "labels", "val"))

print("✅ Dataset prepared successfully.")

✅ Dataset prepared successfully.


In [ ]:
yaml_content = """
train: /content/dataset_yolo/images/train
val: /content/dataset_yolo/images/val
nc: 10  # Number of classes
names: ['background', 'accessories', 'bag', 'belt', 'blazer', 'blouse', 'bodysuit', 'boots', 'bra', 'bracelet']
"""

yaml_path = "/content/clothing-segmentation.yaml"
with open(yaml_path, "w") as file:
    file.write(yaml_content)

print("✅ YAML file created successfully.")

✅ YAML file created successfully.


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.3/949.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 67.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt")
results = model.train(data=yaml_path, epochs=50, imgsz=640, batch=8)

print("✅ Training completed successfully.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 5.90M/5.90M [00:00<00:00, 74.4MB/s]


Ultralytics 8.3.93 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=segment, mode=train, model=yolo11n-seg.pt, data=/content/clothing-segmentation.yaml, epochs=50, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=Tru

100%|██████████| 755k/755k [00:00<00:00, 14.2MB/s]


Overriding model.yaml nc=80 with nc=10

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytic

...


50 epochs completed in 0.023 hours.
Optimizer stripped from runs/segment/train/weights/last.pt, 6.0MB
Optimizer stripped from runs/segment/train/weights/best.pt, 6.0MB

Validating runs/segment/train/weights/best.pt...
Ultralytics 8.3.93 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n-seg summary (fused): 113 layers, 2,836,518 parameters, 0 gradients, 10.2 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00, 10.05it/s]
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:491: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:526: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.05)
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:526: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.05)
/usr/local

                   all         11          0          0          0          0          0          0          0          0          0
WARNING ⚠️ no labels found in segment set, can not compute metrics without labels


/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:654: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index


Speed: 0.2ms preprocess, 2.5ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs/segment/train
✅ Training completed successfully.


In [ ]:
metrics = model.val()
print("Validation Metrics:", metrics)

results = model.predict(source="/content/dataset_yolo/images/val", save=True)

Ultralytics 8.3.93 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n-seg summary (fused): 113 layers, 2,836,518 parameters, 0 gradients, 10.2 GFLOPs


val: Scanning /content/dataset_yolo/labels/val.cache... 0 images, 11 backgrounds, 0 corrupt: 100%|██████████| 11/11 [00:00<?, ?it/s]

WARNING ⚠️ No labels found in /content/dataset_yolo/labels/val.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:00<00:00,  3.57it/s]
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:491: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:526: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.05)
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:526: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.05)
/usr/loca

                   all         11          0          0          0          0          0          0          0          0          0
WARNING ⚠️ no labels found in segment set, can not compute metrics without labels


/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:654: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index


Speed: 0.2ms preprocess, 38.2ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to runs/segment/train2
Validation Metrics: ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([], dtype=int64)
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d378fb8a750>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     

In [ ]:
import os
import cv2
import numpy as np

val_images_dir = "/content/dataset_yolo/images/val"
val_masks_dir = "/content/dataset_yolo/labels/val"

val_images = sorted(os.listdir(val_images_dir))
val_masks = sorted(os.listdir(val_masks_dir))

print("Validation Images:", val_images[:5])
print("Validation Masks:", val_masks[:5])

Validation Images: ['img_0003.jpeg', 'img_0005.jpeg', 'img_0012.jpeg', 'img_0013.jpeg', 'img_0025.jpeg']
Validation Masks: ['img_0003.png', 'img_0005.png', 'img_0012.png', 'img_0013.png', 'img_0025.png']


In [ ]:
import matplotlib.pyplot as plt

def visualize_results(image_path, mask_path, model):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    results = model(image_path)

    if results[0].masks is not None:
        pred_mask = results[0].masks.data[0].cpu().numpy()
        pred_mask = (pred_mask * 255).astype(np.uint8)
    else:
        print(f"Warning: No masks found for image {image_path}. Skipping visualization.")
        return

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(image)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(pred_mask, cmap="jet")
    plt.title("Predicted Mask")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(gt_mask, cmap="jet")
    plt.title("Ground Truth Mask")
    plt.axis("off")

    plt.show()

for i in range(5):
    image_path = os.path.join(val_images_dir, val_images[i])
    mask_path = os.path.join(val_masks_dir, val_masks[i])
    visualize_results(image_path, mask_path, model)


image 1/1 /content/dataset_yolo/images/val/img_0003.jpeg: 640x448 (no detections), 21.0ms
Speed: 3.5ms preprocess, 21.0ms inference, 6.8ms postprocess per image at shape (1, 3, 640, 448)

image 1/1 /content/dataset_yolo/images/val/img_0005.jpeg: 640x448 (no detections), 23.2ms
Speed: 11.4ms preprocess, 23.2ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 448)

image 1/1 /content/dataset_yolo/images/val/img_0012.jpeg: 640x448 (no detections), 29.5ms
Speed: 10.4ms preprocess, 29.5ms inference, 3.6ms postprocess per image at shape (1, 3, 640, 448)

image 1/1 /content/dataset_yolo/images/val/img_0013.jpeg: 640x448 (no detections), 24.8ms
Speed: 2.8ms preprocess, 24.8ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 448)

image 1/1 /content/dataset_yolo/images/val/img_0025.jpeg: 640x448 (no detections), 39.5ms
Speed: 13.0ms preprocess, 39.5ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 448)


In [ ]:
import os
import cv2
import numpy as np

mask_dir = "/content/dataset/jpeg_masks/MASKS"
output_dir = "/content/dataset_yolo/labels"
os.makedirs(output_dir, exist_ok=True)

def convert_mask_to_yolo_format(mask_path, output_path, class_id=0):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    with open(output_path, "w") as file:
        for contour in contours:
            epsilon = 0.002 * cv2.arcLength(contour, True)
            polygon = cv2.approxPolyDP(contour, epsilon, True)

            polygon = polygon.reshape(-1, 2)
            polygon = polygon / np.array([mask.shape[1], mask.shape[0]])

            file.write(f"{class_id} ")
            file.write(" ".join([f"{x:.6f} {y:.6f}" for x, y in polygon]))
            file.write("\n")

for mask_name in os.listdir(mask_dir):
    if mask_name.endswith(".jpeg"):
        mask_path = os.path.join(mask_dir, mask_name)
        output_path = os.path.join(output_dir, mask_name.replace(".jpeg", ".txt"))
        convert_mask_to_yolo_format(mask_path, output_path)
        print(f"✅ Converted {mask_name} to YOLOv8 format.")

print("🎉 All masks converted successfully.")

✅ Converted seg_0233.jpeg to YOLOv8 format.
✅ Converted seg_0973.jpeg to YOLOv8 format.
✅ Converted seg_0380.jpeg to YOLOv8 format.
✅ Converted seg_0818.jpeg to YOLOv8 format.
✅ Converted seg_0917.jpeg to YOLOv8 format.
✅ Converted seg_0105.jpeg to YOLOv8 format.
✅ Converted seg_0446.jpeg to YOLOv8 format.
✅ Converted seg_0822.jpeg to YOLOv8 format.
✅ Converted seg_0226.jpeg to YOLOv8 format.
✅ Converted seg_0503.jpeg to YOLOv8 format.
✅ Converted seg_0400.jpeg to YOLOv8 format.
✅ Converted seg_0536.jpeg to YOLOv8 format.
✅ Converted seg_0253.jpeg to YOLOv8 format.
✅ Converted seg_0680.jpeg to YOLOv8 format.
✅ Converted seg_0672.jpeg to YOLOv8 format.
✅ Converted seg_0678.jpeg to YOLOv8 format.
✅ Converted seg_0098.jpeg to YOLOv8 format.
✅ Converted seg_0349.jpeg to YOLOv8 format.
✅ Converted seg_0689.jpeg to YOLOv8 format.
✅ Converted seg_0963.jpeg to YOLOv8 format.
✅ Converted seg_0039.jpeg to YOLOv8 format.
✅ Converted seg_0483.jpeg to YOLOv8 format.
✅ Converted seg_0395.jpeg to YOL

In [ ]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

image_dir = "/content/dataset/jpeg_images/IMAGES"
mask_dir = "/content/dataset_yolo/labels"
label_path = "/content/dataset/labels.csv"

dataset_path = "/content/dataset_yolo"
os.makedirs(dataset_path, exist_ok=True)
os.makedirs(os.path.join(dataset_path, "images", "train"), exist_ok=True)
os.makedirs(os.path.join(dataset_path, "images", "val"), exist_ok=True)
os.makedirs(os.path.join(dataset_path, "labels", "train"), exist_ok=True)
os.makedirs(os.path.join(dataset_path, "labels", "val"), exist_ok=True)

labels_df = pd.read_csv(label_path)
labels_df["label_list"] = labels_df["label_list"].fillna("background")

train_df, val_df = train_test_split(labels_df, test_size=0.2, random_state=42)

def prepare_dataset(df, image_source, mask_source, image_dest, label_dest):
    for _, row in df.iterrows():
        image_name = f"img_{row['Unnamed: 0']:04d}.jpeg"
        mask_name = f"seg_{row['Unnamed: 0']:04d}.txt"

        src_image_path = os.path.join(image_source, image_name)
        if not os.path.exists(src_image_path):
            print(f"Warning: Image {src_image_path} not found. Skipping.")
            continue
        shutil.copy(src_image_path, os.path.join(image_dest, image_name))

        src_mask_path = os.path.join(mask_source, mask_name)
        if not os.path.exists(src_mask_path):
            print(f"Warning: Mask {src_mask_path} not found. Skipping.")
            continue
        shutil.copy(src_mask_path, os.path.join(label_dest, mask_name))

prepare_dataset(train_df, image_dir, mask_dir,
                os.path.join(dataset_path, "images", "train"),
                os.path.join(dataset_path, "labels", "train"))

prepare_dataset(val_df, image_dir, mask_dir,
                os.path.join(dataset_path, "images", "val"),
                os.path.join(dataset_path, "labels", "val"))

print("✅ Dataset prepared successfully.")

✅ Dataset prepared successfully.


In [ ]:
class_names = labels_df["label_list"].tolist()

yaml_content = f"""
train: /content/dataset_yolo/images/train
val: /content/dataset_yolo/images/val
nc: {len(class_names)}  # Number of classes
names: {class_names}  # List of class names
"""

yaml_path = "/content/clothing-segmentation.yaml"
with open(yaml_path, "w") as file:
    file.write(yaml_content)

print("✅ YAML file created successfully.")
print(yaml_content)

✅ YAML file created successfully.

train: /content/dataset_yolo/images/train
val: /content/dataset_yolo/images/val
nc: 59  # Number of classes
names: ['background', 'accessories', 'bag', 'belt', 'blazer', 'blouse', 'bodysuit', 'boots', 'bra', 'bracelet', 'cape', 'cardigan', 'clogs', 'coat', 'dress', 'earrings', 'flats', 'glasses', 'gloves', 'hair', 'hat', 'heels', 'hoodie', 'intimate', 'jacket', 'jeans', 'jumper', 'leggings', 'loafers', 'necklace', 'panties', 'pants', 'pumps', 'purse', 'ring', 'romper', 'sandals', 'scarf', 'shirt', 'shoes', 'shorts', 'skin', 'skirt', 'sneakers', 'socks', 'stockings', 'suit', 'sunglasses', 'sweater', 'sweatshirt', 'swimwear', 't-shirt', 'tie', 'tights', 'top', 'vest', 'wallet', 'watch', 'wedges']  # List of class names



In [ ]:
from ultralytics import YOLO

model1 = YOLO("yolo11n-seg.pt")
results = model1.train(data=yaml_path, epochs=50, imgsz=640, batch=8, verbose=True)

print("✅ Training completed successfully.")

Ultralytics 8.3.93 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=segment, mode=train, model=yolo11n-seg.pt, data=/content/clothing-segmentation.yaml, epochs=50, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train4, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=Tr

train: Scanning /content/dataset_yolo/labels/train.cache... 0 images, 47 backgrounds, 0 corrupt: 100%|██████████| 47/47 [00:00<?, ?it/s]

WARNING ⚠️ No labels found in /content/dataset_yolo/labels/train.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



val: Scanning /content/dataset_yolo/labels/val.cache... 0 images, 11 backgrounds, 0 corrupt: 100%|██████████| 11/11 [00:00<?, ?it/s]

WARNING ⚠️ No labels found in /content/dataset_yolo/labels/val.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.


Plotting labels to runs/segment/train4/labels.jpg... 
zero-size array to reduction operation maximum which has no identity
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000159, momentum=0.9) with parameter groups 90 weight(decay=0.0), 101 weight(decay=0.0005), 100 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to runs/segment/train4
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


       1/50      1.88G          0          0      63.52          0          0        640: 100%|██████████| 6/6 [00:02<00:00,  2.58it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00, 11.86it/s]


RuntimeError: torch.cat(): expected a non-empty list of Tensors

In [ ]:
val_images_dir = "/content/dataset_yolo/images/val"
val_masks_dir = "/content/dataset_yolo/labels/val"

val_images = os.listdir(val_images_dir)
val_masks = os.listdir(val_masks_dir)

print("Validation Images:", val_images[:10])
print("Validation Masks:", val_masks[:10])

Validation Images: ['img_0003.jpeg', 'img_0045.jpeg', 'img_0049.jpeg', 'img_0034.jpeg', 'img_0047.jpeg', 'img_0025.jpeg', 'img_0013.jpeg', 'img_0057.jpeg', 'img_0053.jpeg', 'img_0012.jpeg']
Validation Masks: ['img_0013.png', 'seg_0012.txt', 'img_0049.png', 'img_0003.png', 'img_0034.png', 'seg_0003.txt', 'img_0012.png', 'seg_0053.txt', 'img_0057.png', 'seg_0025.txt']


In [1]:
import os
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

image_dir = "/content/dataset/jpeg_masks/MASKS"
label_dir = "/content/dataset_yolo/labels"

def visualize_mask(image_path, label_path):
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error loading image: {image_path}")
        return

    height, width, _ = image.shape

    try:
        with open(label_path, "r") as file:
            lines = file.readlines()
            for line in lines:
                data = line.strip().split()
                class_id = int(data[0])
                points = np.array(data[1:], dtype=np.float32).reshape(-1, 2)

                points[:, 0] *= width
                points[:, 1] *= height
                points = points.astype(np.int32)

                cv2.polylines(image, [points], isClosed=True, color=(0, 255, 0), thickness=2)
                cv2.putText(image, f"Class: {class_id}", (points[0][0], points[0][1] - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    except Exception as e:
        print(f"Error reading label: {label_path} - {e}")

    cv2_imshow(image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

for image_name in os.listdir(image_dir):
    if image_name.endswith(".jpg") or image_name.endswith(".jpeg"):
        label_name = image_name.replace(".jpg", ".txt").replace(".jpeg", ".txt")
        image_path = os.path.join(image_dir, image_name)
        label_path = os.path.join(label_dir, label_name)

        if os.path.exists(label_path):
            print(f"Visualizing {image_name}")
            visualize_mask(image_path, label_path)
        else:
            print(f"Label not found for {image_name}")



FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/jpeg_masks/MASKS'

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

def visualize_results(image_path, mask_path, model):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    results = model(image_path)
    if results[0].masks is not None:
        pred_mask = results[0].masks.data[0].cpu().numpy()
        pred_mask = (pred_mask * 255).astype(np.uint8)
    else:
        pred_mask = np.zeros_like(gt_mask)

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(image)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(pred_mask, cmap="jet")
    plt.title("Predicted Mask")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(gt_mask, cmap="jet")
    plt.title("Ground Truth Mask")
    plt.axis("off")

    plt.show()

val_images_dir = os.path.join(dataset_path, "images", "val")
val_masks_dir = os.path.join(dataset_path, "labels", "val")
val_images = sorted(os.listdir(val_images_dir))
val_masks = sorted(os.listdir(val_masks_dir))

for i in range(5):
    image_path = os.path.join(val_images_dir, val_images[i])
    mask_path = os.path.join(val_masks_dir, val_masks[i])
    visualize_results(image_path, mask_path, model1)